# Dual-regression-like spatial modes from resting-state fMRI

For $N$ subjects with resting-state fMRI (CIFTI), each subject's data
$C_s$ (vertices x time) is factorised twice in a single joint decomposition:

$$C_s = A_{\text{group}} S_s^T \quad\text{and}\quad C_s = A_s S_s^T.$$

The shared timecourse $S_s$ links the group spatial map $A_{\text{group}}$
to each subject-specific map $A_s$, analogous to dual regression but solved
jointly in a single optimisation.

In [ ]:
import os, sys, pickle
from pathlib import Path
import numpy as np
import nibabel as nib

sys.path.insert(0, os.path.abspath('..'))
from pathfinder.decomp import JointOuterDecomp

DATA_ROOT = '/well/win-biobank/users/gbb787/analyzePRF_HCP7TRET'
SUBJECT_IDS = np.loadtxt('sublist7T3T.txt', dtype=str)[:100]
REFERENCE_CIFTI = 'GroupAcg.ecc_scaled.dscalar.nii'
TASKS = ['REST1_LR', 'REST1_RL', 'REST2_LR', 'REST2_RL']

N_COMPONENTS = 25
N_ITER = 20
ALPHA_REG = 0.1
BATCH_SIZE = 120

OUT_DIR = Path('spatial_maps_rest25')
OUT_DIR.mkdir(exist_ok=True, parents=True)

## Load and concatenate resting-state runs

Each subject's 4 REST runs are demeaned and concatenated along time.

In [ ]:
def load_cifti(subject_id, task):
    path = Path(DATA_ROOT) / subject_id / f'rfMRI_{task}_Atlas_MSMAll_hp2000_clean_2mm.dtseries.nii'
    data = nib.load(str(path)).get_fdata().T  # (vertices, time)
    data -= data.mean(axis=1, keepdims=True)
    return data


subject_data = {}
for sid in SUBJECT_IDS:
    runs = []
    for task in TASKS:
        try:
            runs.append(load_cifti(sid, task))
        except FileNotFoundError:
            print(f'  {sid}/{task} missing, skipping')
    if runs:
        subject_data[sid] = np.hstack(runs)

subject_ids = sorted(subject_data.keys())
N = len(subject_ids)
print(f'Loaded {N} subjects')

## Build the joint factorisation

Each subject appears twice in `Clist`:

* a **group copy** with $\alpha = 0$ (shared group spatial mode),
* an **individual copy** with $\alpha = i+1$ (subject-specific spatial mode).

Both copies share the same $\beta = i$ so the subject's temporal factor is
jointly constrained by the subject's and the group's data.

In [ ]:
Clist, alpha_map, beta_map = [], [], []
for i, sid in enumerate(subject_ids):
    Clist.append(subject_data[sid]); alpha_map.append(0);     beta_map.append(i)
for i, sid in enumerate(subject_ids):
    Clist.append(subject_data[sid]); alpha_map.append(i + 1); beta_map.append(i)

print(f'{len(Clist)} matrices, {max(alpha_map)+1} spatial modes '
      f'(1 group + {N} individual), {max(beta_map)+1} temporal modes')

## Fit PathFinder

In [ ]:
model = JointOuterDecomp(
    n_components=N_COMPONENTS, n_iter=N_ITER, alpha=ALPHA_REG,
    batch_size=BATCH_SIZE, init_type='svd_quick', do_ica='both', verbose=True,
)
model.fit(Clist, alpha=alpha_map, beta=beta_map)
print(f'Final loss: {np.mean(model._loss[-1]):.4f}')

## Extract group and individual spatial maps

In [ ]:
group_spatial = model._A[0]
individual_spatial = {sid: model._A[i + 1] for i, sid in enumerate(subject_ids)}
timecourses = {sid: model._S[i] for i, sid in enumerate(subject_ids)}

print(f'Group spatial map: {group_spatial.shape}')
for sid in subject_ids[:5]:
    r = np.mean([np.corrcoef(group_spatial[:, c], individual_spatial[sid][:, c])[0, 1]
                 for c in range(N_COMPONENTS)])
    print(f'  {sid}: mean corr with group = {r:.3f}')

## Save as CIFTI dscalar + pickle

In [ ]:
ref_cifti = nib.load(str(REFERENCE_CIFTI))
brain_axis = ref_cifti.header.get_axis(1)
scalar_axis = nib.cifti2.ScalarAxis([f'Component_{i+1}' for i in range(N_COMPONENTS)])
header = nib.cifti2.Cifti2Header.from_axes((scalar_axis, brain_axis))


def save_cifti(name, data):
    img = nib.Cifti2Image(data.T, header)  # CIFTI expects (components, vertices)
    nib.save(img, str(OUT_DIR / f'spatial_map_{name}.dscalar.nii'))


save_cifti('group', group_spatial)
for sid in subject_ids:
    save_cifti(sid, individual_spatial[sid])

with open(OUT_DIR / 'timeseries.pkl', 'wb') as f:
    pickle.dump(timecourses, f)

np.savez(OUT_DIR / 'results.npz',
         group_spatial=group_spatial,
         subject_ids=np.array(subject_ids),
         loss=np.array(model._loss),
         **{f'A_{sid}': individual_spatial[sid] for sid in subject_ids},
         **{f'S_{sid}': timecourses[sid] for sid in subject_ids})
print(f'Saved to {OUT_DIR}/')